In [ ]:
!pip install -U transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import gc
import os
import re
import sqlite3
import time

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Point this at mounted Drive, e.g. "/content/drive/MyDrive/tinyllama-sql"
OUTPUT_ROOT = "./tinyllama-sql-runs"

# Caps train size so each run finishes in a reasonable time on a T4.
# Raise it later if you have more session time or a better GPU.
TRAIN_SUBSET_SIZE = 1500
EVAL_ACCURACY_SAMPLES = 100

LORA_CONFIG_KWARGS = dict(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
raw_dataset = load_dataset("b-mc2/sql-create-context")
split = raw_dataset["train"].train_test_split(test_size=0.03, seed=42)
train_raw_full, eval_raw = split["train"], split["test"]

train_raw = train_raw_full.select(range(min(TRAIN_SUBSET_SIZE, len(train_raw_full))))
print(f"Train examples: {len(train_raw)} | Eval examples: {len(eval_raw)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token


def format_example(example):
    messages = [
        {
            "role": "user",
            "content": (
                f"Given this database schema:\n{example['context'].strip()}\n\n"
                f"Write a SQL query to answer: {example['question'].strip()}"
            ),
        },
        {"role": "assistant", "content": example["answer"].strip()},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}


train_dataset = train_raw.map(format_example)
eval_dataset = eval_raw.map(format_example)

print("--- Sample formatted training example ---")
print(train_dataset["text"][0])

In [ ]:
def build_sqlite_db(create_statements: str) -> sqlite3.Connection:
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    for stmt in re.split(r";\s*(?=CREATE)", create_statements.strip()):
        stmt = stmt.strip().rstrip(";")
        if stmt:
            try:
                cur.execute(stmt)
            except sqlite3.OperationalError:
                pass
    conn.commit()
    return conn


def run_sql_safely(conn: sqlite3.Connection, sql: str):
    try:
        cur = conn.cursor()
        cur.execute(sql)
        return cur.fetchall()
    except Exception:
        return None


def generate_sql(question: str, context: str, inference_model) -> str:
    messages = [
        {
            "role": "user",
            "content": (
                f"Given this database schema:\n{context}\n\n"
                f"Write a SQL query to answer: {question}"
            ),
        }
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(inference_model.device)
    with torch.no_grad():
        output = inference_model.generate(
            **inputs, max_new_tokens=128, do_sample=False
        )
    return tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


def evaluate_execution_accuracy(dataset, inference_model, n_samples: int) -> float:
    n_samples = min(n_samples, len(dataset))
    matches = 0
    for i in range(n_samples):
        example = dataset[i]
        conn = build_sqlite_db(example["context"])
        predicted_sql = generate_sql(example["question"], example["context"], inference_model)
        gold_result = run_sql_safely(conn, example["answer"])
        pred_result = run_sql_safely(conn, predicted_sql)
        if gold_result is not None and pred_result is not None and gold_result == pred_result:
            matches += 1
        conn.close()
    return matches / n_samples

In [ ]:
def run_experiment(use_qlora: bool) -> dict:
    run_name = "qlora" if use_qlora else "lora"
    output_dir = os.path.join(OUTPUT_ROOT, run_name)
    print(f"\n{'=' * 60}\nStarting {run_name.upper()} run -> {output_dir}\n{'=' * 60}")

    if use_qlora:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, # Changed to bfloat16
            bnb_4bit_use_double_quant=True,
        )
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=bnb_config, device_map="auto",
            dtype=torch.bfloat16, # Changed to bfloat16
        )
        # Required standard step for QLoRA: casts layer norms to fp32 and
        # correctly configures the quantized model for gradient computation.
        # Skipping this is what caused the BFloat16/GradScaler error.
        base_model = prepare_model_for_kbit_training(base_model)
        precision_args = {"bf16": True, "fp16": False} # Set precision for QLoRA
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, device_map="auto", dtype=torch.float16,
        )
        precision_args = {"bf16": False, "fp16": True} # Set precision for LoRA

    lora_config = LoraConfig(**LORA_CONFIG_KWARGS)
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()

    peak_mem_before_train = torch.cuda.max_memory_allocated() / 1024**3

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        gradient_checkpointing=True,  # Set to True to save memory
        learning_rate=2e-4,
        num_train_epochs=2,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=25,
        save_steps=25,
        save_total_limit=2,
        **precision_args,
        report_to="none",
        remove_unused_columns=False,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args,
    )

    last_checkpoint = None
    if os.path.isdir(output_dir):
        checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
        if checkpoints:
            # If this is the QLoRA run, force a fresh start due to potential scaler issues with old checkpoints.
            # Otherwise, try to resume for LoRA.
            if use_qlora:
                print(f"Ignoring existing QLoRA checkpoint due to potential scaler mismatch: {output_dir}")
                last_checkpoint = None
            else:
                last_checkpoint = os.path.join(
                    output_dir, sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
                )
                print(f"Resuming from checkpoint: {last_checkpoint}")

    start_time = time.time()
    trainer.train(resume_from_checkpoint=last_checkpoint)
    train_seconds = time.time() - start_time

    peak_mem_gb = torch.cuda.max_memory_allocated() / 1024**3

    model.save_pretrained(f"{output_dir}/adapter")
    tokenizer.save_pretrained(f"{output_dir}/adapter")

    model.eval()
    accuracy = evaluate_execution_accuracy(eval_raw, model, EVAL_ACCURACY_SAMPLES)
    print(f"{run_name.upper()} execution accuracy: {accuracy:.2%}")

    trainable_params = model.num_parameters(only_trainable=True)
    total_params = model.num_parameters()

    results = {
        "run": run_name,
        "train_minutes": round(train_seconds / 60, 1),
        "peak_vram_gb": round(peak_mem_gb, 2),
        "execution_accuracy": accuracy,
        "trainable_params": trainable_params,
        "total_params": total_params,
        "adapter_path": f"{output_dir}/adapter",
    }

    # --- Free GPU memory before the next run starts ---
    del trainer, model, base_model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    return results

In [ ]:
all_results = []
all_results.append(run_experiment(use_qlora=True))
all_results.append(run_experiment(use_qlora=False))


In [ ]:
print(f"\n{'=' * 60}\nLoRA vs QLoRA comparison\n{'=' * 60}")
header = f"{'Run':<8} {'Train (min)':<13} {'Peak VRAM (GB)':<16} {'Exec accuracy':<15}"
print(header)
print("-" * len(header))
for r in all_results:
    print(f"{r['run']:<8} {r['train_minutes']:<13} {r['peak_vram_gb']:<16} {r['execution_accuracy']:.2%}")